### Pydantic - provides validation

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1113bcdd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11151d050>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [6]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1113bcdd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11151d050>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type

In [7]:
model.invoke("Provide details about movie Dhurandhar")

AIMessage(content='<think>\nOkay, so the user is asking about the movie "Dhurandhar." First, I need to recall what I know about this film. Let me think... I remember it\'s a Bollywood movie, right? It\'s a comedy, maybe from the 2000s. The title "Dhurandhar" makes me think of a character name.\n\nI think the main actor is probably someone well-known. Wait, wasn\'t it starring Akshay Kumar? Or maybe Shah Rukh Khan? Hmm, no, Shah Rukh did "Dilwale Dulhania Le Jayenge," but not this one. Akshay Kumar has done a lot of comedies, so maybe he\'s the lead. Let me confirm that. \n\nThe director might be a name like Priyadarshan or maybe someone else. Priyadarshan directed a lot of comedies, so that\'s possible. The plot: the user might want to know about the storyline. From what I remember, the movie is about a man named Dhurandhar, who is clumsy or not the sharpest tool in the shed. He stumbles upon a conspiracy or something related to a gold smuggling ring. His innocence helps him expose the

In [9]:
response=model_with_structure.invoke("Provide details about the moview 3 idiots")
response

Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8.4)

### Message output alongside parsed structure

In [11]:

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie 3 idiots")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "3 Idiots." Let me see what I need to do. The available tool is the Movie function, which requires title, year, director, and rating. First, I need to recall the information about "3 Idiots." The title is definitely "3 Idiots." It was released in 2009, I think. The director is Rajkumar Hirani. As for the rating, it\'s a popular movie, maybe around 8.4 on IMDb. Wait, I should confirm these details to make sure. Let me check: yes, the release year is 2009, directed by Rajkumar Hirani, and the IMDb rating is 8.4. So I need to structure the function call with these parameters. Make sure all required fields are included and the data types are correct. The year should be an integer, the rating a number. Alright, that should do it.\n', 'tool_calls': [{'id': 'g8ck05q9d', 'function': {'arguments': '{"director":"Rajkumar Hirani","rating":8.4,"title":"3 Idiots","year":2009}'

### Nested structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float = Field(..., description="The movie's budget in millions usd")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie 3 idiots")
response

MovieDetails(title='3 Idiots', year=2009, cast=[Actor(name='Aamir Khan', role='Raju Rastogi'), Actor(name='R. Madhavan', role='Farhan Qureshi'), Actor(name='Sharman Joshi', role='Rane Iyer'), Actor(name='Boman Irani', role='Professor Viru Sahastrabuddhe'), Actor(name='Kona Kanchan', role='Pia')], genres=['Comedy', 'Drama', 'Satire'], budget=10.0)

### TypedDict -used when no need of validation

In [13]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

### Response format in create agent(pydantic)

In [14]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str = Field(description="Name of the person")
    email: str = Field(description="Email address of the person")
    phone: str = Field(description="phone no of person")

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages": [{"role":"user","content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='8c9f625a-db2f-4204-bcd7-69b04a71e783'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters for name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is "John Doe", which is straightforward. The email is "john@example.com". The phone number is "(555) 123-4567". I should check if the phone number is in the correct format. The function expects a string for each field. The phone number here includes parentheses and a space, but since the tool\'s parameter type is a string, I can include it as-is without formatting changes. \n\nI need to make sure all required fields are present. The inpu

### Dataclasses

In [16]:


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')